# Google Drive Uploader Worker
This notebook continuously monitors a queue file and downloads videos/files to Google Drive.

**Instructions:**
1. Run Cell 1 to mount Drive and install dependencies
2. Run Cell 2 to configure paths
3. Run Cell 3 to start the worker loop (runs until manually stopped)

In [ ]:
# Cell 1: Setup - Mount Google Drive and Install Dependencies

from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')
print("✓ Drive mounted successfully")

print("\nInstalling yt-dlp...")
%pip install -q yt-dlp requests
print("✓ yt-dlp installed successfully")

print("\n" + "="*50)
print("Setup complete! Proceed to Cell 2")
print("="*50)

In [ ]:
# Cell 2: Configuration - Define Paths and Settings

import json
import os

# Base paths
DRIVE_BASE = "/content/drive/MyDrive"
UPLOADER_DIR = os.path.join(DRIVE_BASE, ".uploader")
QUEUE_FILE = os.path.join(UPLOADER_DIR, "queue.json")
STATUS_FILE = os.path.join(UPLOADER_DIR, "status.json")

# Worker settings
POLL_INTERVAL = 5  # seconds between queue checks
MAX_RETRIES = 3    # retry attempts for failed downloads

# Create .uploader directory if it doesn't exist
os.makedirs(UPLOADER_DIR, exist_ok=True)
print(f"✓ Uploader directory: {UPLOADER_DIR}")

# Initialize queue file if it doesn't exist
if not os.path.exists(QUEUE_FILE):
    with open(QUEUE_FILE, 'w') as f:
        json.dump({"downloads": []}, f, indent=2)
    print(f"✓ Created queue file: {QUEUE_FILE}")
else:
    print(f"✓ Queue file exists: {QUEUE_FILE}")

# Initialize status file if it doesn't exist
if not os.path.exists(STATUS_FILE):
    with open(STATUS_FILE, 'w') as f:
        json.dump({"downloads": {}, "last_updated": ""}, f, indent=2)
    print(f"✓ Created status file: {STATUS_FILE}")
else:
    print(f"✓ Status file exists: {STATUS_FILE}")

print("\n" + "="*50)
print("Configuration complete! Proceed to Cell 3 to start worker")
print("="*50)

In [ ]:
# Cell 3: Worker Loop - Process Downloads from Queue

import json
import os
import time
from datetime import datetime

import requests
import yt_dlp

# Global variables for tracking
current_download = None

def load_queue():
    """Load the queue from JSON file."""
    try:
        with open(QUEUE_FILE) as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading queue: {e}")
        return {"downloads": []}

def save_queue(queue_data):
    """Save the queue to JSON file."""
    try:
        with open(QUEUE_FILE, 'w') as f:
            json.dump(queue_data, f, indent=2)
    except Exception as e:
        print(f"Error saving queue: {e}")

def update_status(download_id, status_data):
    """Update the status file for a specific download."""
    try:
        # Load current status
        if os.path.exists(STATUS_FILE):
            with open(STATUS_FILE) as f:
                status = json.load(f)
        else:
            status = {"downloads": {}, "last_updated": ""}

        # Update status for this download
        status["downloads"][download_id] = status_data
        status["last_updated"] = datetime.now().isoformat()

        # Save status
        with open(STATUS_FILE, 'w') as f:
            json.dump(status, f, indent=2)
    except Exception as e:
        print(f"Error updating status: {e}")

def progress_hook(d):
    """yt-dlp progress hook to update status in real-time."""
    global current_download

    if current_download is None:
        return

    download_id = current_download

    if d['status'] == 'downloading':
        # Extract progress information
        percent = d.get('_percent_str', '0%').strip()
        speed = d.get('_speed_str', 'N/A').strip()
        filename = os.path.basename(d.get('filename', 'unknown'))

        # Update status
        update_status(download_id, {
            "status": "downloading",
            "percent": percent,
            "speed": speed,
            "filename": filename
        })

    elif d['status'] == 'finished':
        filename = os.path.basename(d.get('filename', 'unknown'))
        update_status(download_id, {
            "status": "processing",
            "percent": "100%",
            "speed": "N/A",
            "filename": filename
        })

def download_with_ytdlp(url, output_path, download_id):
    """
    Download video using yt-dlp.
    Supports 1500+ video sites including YouTube, Vimeo, etc.
    """
    global current_download
    current_download = download_id

    try:
        # yt-dlp options
        ydl_opts = {
            'outtmpl': os.path.join(output_path, '%(title)s.%(ext)s'),
            'progress_hooks': [progress_hook],
            'format': 'best',
            'quiet': False,
            'no_warnings': False,
        }

        print(f"  Starting yt-dlp download: {url}")

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            filename = ydl.prepare_filename(info)

        print(f"  ✓ Download completed: {os.path.basename(filename)}")

        # Final status update
        update_status(download_id, {
            "status": "completed",
            "percent": "100%",
            "speed": "N/A",
            "filename": os.path.basename(filename)
        })

        current_download = None
        return True, os.path.basename(filename)

    except Exception as e:
        print(f"  ✗ yt-dlp error: {str(e)}")
        current_download = None
        return False, str(e)

def download_direct(url, output_path, download_id):
    """
    Direct download using requests for non-video URLs.
    Fallback method for direct file downloads.
    """
    try:
        print(f"  Starting direct download: {url}")

        # Start download with streaming
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()

        # Get filename from URL or Content-Disposition header
        filename = None
        if 'Content-Disposition' in response.headers:
            content_disp = response.headers['Content-Disposition']
            if 'filename=' in content_disp:
                filename = content_disp.split('filename=')[1].strip('"')

        if not filename:
            filename = os.path.basename(url.split('?')[0]) or 'download'

        filepath = os.path.join(output_path, filename)

        # Get total size
        total_size = int(response.headers.get('content-length', 0))

        # Download with progress tracking
        downloaded = 0
        start_time = time.time()

        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)

                    # Update progress every 100KB
                    if downloaded % (100 * 1024) < 8192:
                        elapsed = time.time() - start_time
                        speed = downloaded / elapsed if elapsed > 0 else 0
                        speed_str = f"{speed / (1024*1024):.2f} MB/s"

                        if total_size > 0:
                            percent = int((downloaded / total_size) * 100)
                            percent_str = f"{percent}%"
                        else:
                            percent_str = f"{downloaded / (1024*1024):.2f} MB"

                        update_status(download_id, {
                            "status": "downloading",
                            "percent": percent_str,
                            "speed": speed_str,
                            "filename": filename
                        })

        print(f"  ✓ Download completed: {filename}")

        # Final status update
        update_status(download_id, {
            "status": "completed",
            "percent": "100%",
            "speed": "N/A",
            "filename": filename
        })

        return True, filename

    except Exception as e:
        print(f"  ✗ Direct download error: {str(e)}")
        return False, str(e)

def process_download(item):
    """
    Process a single download item from the queue.
    Tries yt-dlp first, falls back to direct download.
    """
    download_id = item['id']
    url = item['url']
    folder = item.get('folder', 'Downloads')

    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Processing: {download_id}")
    print(f"  URL: {url}")
    print(f"  Folder: {folder}")

    # Create output directory
    output_path = os.path.join(DRIVE_BASE, folder)
    os.makedirs(output_path, exist_ok=True)

    # Update status to downloading
    update_status(download_id, {
        "status": "downloading",
        "percent": "0%",
        "speed": "N/A",
        "filename": "initializing..."
    })

    # Try yt-dlp first (works for 1500+ video sites)
    success, result = download_with_ytdlp(url, output_path, download_id)

    # If yt-dlp fails, try direct download
    if not success:
        print("  Trying direct download as fallback...")
        success, result = download_direct(url, output_path, download_id)

    # Update final status
    if success:
        item['status'] = 'completed'
        print(f"  ✓ Success: {result}")
    else:
        item['status'] = 'failed'
        update_status(download_id, {
            "status": "failed",
            "percent": "0%",
            "speed": "N/A",
            "filename": f"Error: {result}"
        })
        print(f"  ✗ Failed: {result}")

    return success

def worker_loop():
    """
    Main worker loop that continuously monitors the queue.
    Runs indefinitely until manually stopped (Ctrl+C or Stop button).
    """
    print("="*60)
    print("WORKER STARTED")
    print("="*60)
    print(f"Queue file: {QUEUE_FILE}")
    print(f"Status file: {STATUS_FILE}")
    print(f"Poll interval: {POLL_INTERVAL} seconds")
    print("\nMonitoring queue... (Press Stop button to halt)\n")
    print("="*60)

    iteration = 0

    try:
        while True:
            iteration += 1

            # Load queue
            queue = load_queue()
            downloads = queue.get('downloads', [])

            # Find pending downloads
            pending = [d for d in downloads if d.get('status') == 'pending']

            if pending:
                print(f"\n[Check #{iteration}] Found {len(pending)} pending download(s)")

                # Process each pending download
                for item in pending:
                    try:
                        process_download(item)

                        # Save updated queue
                        save_queue(queue)

                    except Exception as e:
                        print(f"  ✗ Unexpected error: {e}")
                        item['status'] = 'failed'
                        update_status(item['id'], {
                            "status": "failed",
                            "percent": "0%",
                            "speed": "N/A",
                            "filename": f"Error: {str(e)}"
                        })
                        save_queue(queue)
            else:
                # Print heartbeat every 10 iterations
                if iteration % 10 == 0:
                    print(f"[Check #{iteration}] No pending downloads (waiting...)")

            # Wait before next check
            time.sleep(POLL_INTERVAL)

    except KeyboardInterrupt:
        print("\n" + "="*60)
        print("WORKER STOPPED (Keyboard Interrupt)")
        print("="*60)
    except Exception as e:
        print(f"\n✗ Worker error: {e}")
        print("="*60)
        print("WORKER STOPPED (Error)")
        print("="*60)

# Start the worker loop
worker_loop()